# The Workflow in Jupyter
The last sets of tutorials have shown you how to go from a Bayesian fitter to a full SBI model using the provided CLI. This will do the same steps but purely in a notebook!

In [ ]:
from pathlib import Path

from matplotlib import pyplot as plt
from sbi.analysis import pairplot

from mach3sbitools.diagnostics import compare_logl
from mach3sbitools.inference import InferenceHandler
from mach3sbitools.simulator import Simulator
from mach3sbitools.utils import MaCh3Logger, PosteriorConfig, TrainingConfig

MaCh3Logger("mach3sbitools")

First things first we should set up our file paths

In [ ]:
# Some global variables
BASE = Path("jupyter_tutorial")

DATA_FILE = BASE / "data/my_data.feather"
PRIOR_FILE = BASE / "prior/my_prior.pkl"
MODEL_FILE = BASE / "model/my_model.ckpt"
INFERENCE_FILE = BASE / "inference/my_inference.paraquet"

Now we load in the simulations. This is the same as running `mach3sbi simulate` and will produce the same outputs. We also save the prior in this step

In [ ]:
# Firstly we load the simulator
simulator = Simulator(
    module_name="my_simulator",
    class_name="MySimulator",
    config=Path("physics_configs/PhysicsConfig.yaml"),
)

# This should take ~2 mins
theta, x = simulator.simulate(400000)
simulator.save(DATA_FILE, theta, x)

simulator.prior.save(PRIOR_FILE)


We can now train the SBI, this is the same as the `mach3sbi train` step in the CLI

In [ ]:
# We divide our training configuration into two configurations.

# This sets up model properties
posterior_config = PosteriorConfig(
    model="maf",
    hidden_features=64,
    num_transforms=5,
    dropout_probability=0.1,
)

# This sets up training
training_config = TrainingConfig(
    save_path=MODEL_FILE,
    max_epochs=2000,
    learning_rate=1e-4,
    show_progress=True,
    tensorboard_dir=BASE/"tensor-logs",
)

# We can now initialise the density estimator settingss
trainer = InferenceHandler(PRIOR_FILE)
trainer.set_dataset(DATA_FILE.parent)
trainer.load_training_data()
trainer.create_posterior(posterior_config)

In [ ]:
# Now we run training, this will take some time! 
trainer.train_posterior(training_config, model_config=posterior_config)

Finally we run `mach3sbi inference` which will give us our inference data!

In [ ]:
# data_bins = simulator.simulator_wrapper.get_data_bins()
data_bins = simulator.simulator_wrapper.get_data_bins()
samples = trainer.sample_posterior(1_000_000, data_bins)

In [ ]:
pairplot(samples.cpu().numpy(), labels=simulator.simulator_wrapper.get_parameter_names())
plt.show()

In [ ]:
# Finally we do the diagnostics
compare_logl(simulator, trainer, 100_000, n_bins=100)

After this you can run the same diagnostics we performed in the previous tutorial!!

**Congratulations** you've finished the SBI tutorial! You can now run MaCh3SBITools on your own code. MaCh3 simulator wrappers are provided in the `examples` directory and may be used for fitting with.